## 1. Faster R-CNN

**Contents:**
1. What & Why
2. Architecture (Two-Stage)
3. Anchors & the RPN
4. RoIAlign / RoIPool and the Detection Head
5. Loss & Training
6. torchvision Usage
7. References

---

**Faster R-CNN** (Ren, He, Girshick, Sun, NeurIPS 2015) is the classic **two-stage** object detector. It is *accurate but not the fastest* — typically used when precision matters more than real-time speed (medical imaging, fine-grained detection, dense scenes).

It is the third member of the R-CNN family:

- **R-CNN** (2014): run selective search to get ~2000 region proposals, warp each, run a CNN on every crop separately. Extremely slow.
- **Fast R-CNN** (2015): run the CNN **once** over the whole image, then crop features (not pixels) per proposal via **RoIPool**. Much faster, but still relies on an **external** proposal method (selective search) that runs on CPU and dominates runtime.
- **Faster R-CNN** (2015): replaces selective search with a learned **Region Proposal Network (RPN)** that shares the backbone's convolutional features. Now proposals are nearly free and the whole pipeline is end-to-end trainable on GPU.

## 2. Architecture (Two-Stage)

The pipeline is:

```
Image
  |
  v
Backbone CNN (+ FPN)   ->  shared feature maps
  |
  +--> Stage 1: RPN     ->  objectness + box deltas on anchors  ->  ~1000 region proposals
  |
  v
RoIAlign (crop a fixed HxW feature per proposal)
  |
  v
Stage 2: Box Head (FC layers)
  +--> classification: object class (incl. background)
  +--> regression:     refined box coordinates (per class)
  |
  v
Post-processing: score threshold + per-class NMS  ->  final detections
```

**Backbone.** Usually ResNet-50 with a **Feature Pyramid Network (FPN)**. FPN combines deep, semantically strong but coarse features with shallow, high-resolution features, producing a multi-scale pyramid (P2..P6) so that small and large objects are each handled at an appropriate scale.

**Stage 1 (RPN)** proposes *where* objects might be (class-agnostic regions). **Stage 2** decides *what* each region is and refines its box. This division of labour is what "two-stage" means and is the main reason Faster R-CNN is accurate: the second stage only has to reason about a small set of already-promising regions.

## 3. Anchors & the RPN

**Anchors** are a fixed grid of reference boxes tiled densely over the feature map. At each spatial location the RPN places `k` anchors of different **scales** and **aspect ratios** (e.g. 3 scales x 3 ratios = 9 anchors per location). With FPN, each pyramid level uses anchors of one scale, so each level specializes in one object size.

The **RPN** is a tiny conv network applied to every feature location. For each anchor it predicts:

- an **objectness score** (object vs. background), and
- **4 box-regression deltas** `(t_x, t_y, t_w, t_h)` that nudge the anchor toward a ground-truth box.

The deltas are applied relative to the anchor `(x_a, y_a, w_a, h_a)`:

```
x = x_a + t_x * w_a       w = w_a * exp(t_w)
y = y_a + t_y * h_a       h = h_a * exp(t_h)
```

Anchors are labelled positive if IoU with a ground-truth box > 0.7 and negative if IoU < 0.3. The top-scoring proposals are kept, **NMS** removes duplicates, and ~1000–2000 survive into stage 2.

## 4. RoIAlign / RoIPool and the Detection Head

Proposals have arbitrary sizes, but the fully-connected detection head needs a **fixed-size** feature (e.g. 7x7). The cropping operator solves this:

- **RoIPool** (Fast R-CNN) quantizes the proposal boundaries to the feature-map grid, then max-pools into a fixed grid. The two roundings introduce small misalignments.
- **RoIAlign** (introduced with Mask R-CNN, now standard) avoids quantization by sampling features at exact floating-point locations with **bilinear interpolation**. This alignment is important for accurate localization, so torchvision's FPN models use RoIAlign.

The pooled feature goes to the **box head** (two FC layers) which outputs, per class:

- a softmax class score (the `C` foreground classes **plus** a background class), and
- class-specific box-regression deltas to refine the proposal one more time.

## 5. Loss & Training

Both stages use a **multi-task loss** = classification + box regression.

```
L = L_cls + lambda * L_reg
```

- **Classification.** RPN: binary cross-entropy (object vs. background). Box head: cross-entropy over `C + 1` classes.
- **Box regression.** Smooth L1 (Huber) loss on the `(t_x, t_y, t_w, t_h)` deltas, applied **only to positive (foreground) samples**.

To keep training balanced, each stage samples a fixed mini-batch of anchors/proposals with a target positive:negative ratio (e.g. 1:1 for the RPN, 1:3 for the head), since background vastly outnumbers foreground. The whole network is trained jointly end-to-end.

## 6. torchvision Usage

torchvision ships pretrained Faster R-CNN models in `torchvision.models.detection`.

```python
import torch
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights,
)

# Load model pretrained on COCO (91 classes)
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
model.eval()

# The matching preprocessing transform (resize + normalize)
preprocess = weights.transforms()

# Inference: list of CxHxW float tensors in [0, 1]; the model handles resizing
from torchvision.io import read_image
img = read_image("image.jpg")               # uint8 CxHxW
batch = [preprocess(img)]

with torch.no_grad():
    outputs = model(batch)                  # list, one dict per image

pred = outputs[0]
print(pred["boxes"])   # (N, 4) in [x1, y1, x2, y2]
print(pred["labels"])  # (N,) COCO class ids
print(pred["scores"])  # (N,) confidence in [0, 1]

# Map ids to names and filter by confidence
categories = weights.meta["categories"]
keep = pred["scores"] > 0.5
for box, label in zip(pred["boxes"][keep], pred["labels"][keep]):
    print(categories[label], box.tolist())
```

**Training / fine-tuning** uses the same model in `model.train()` mode. Pass images **and** targets and the model returns the loss dict directly:

```python
model.train()
images  = [torch.rand(3, 600, 800)]
targets = [{"boxes": torch.tensor([[10., 20., 200., 300.]]),
            "labels": torch.tensor([1])}]
loss_dict = model(images, targets)   # {'loss_classifier', 'loss_box_reg', 'loss_objectness', 'loss_rpn_box_reg'}
losses = sum(loss_dict.values())
losses.backward()
```

To fine-tune on a custom number of classes, swap the box predictor:

```python
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

num_classes = 5  # your classes + 1 background
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
```

A stronger, more recent variant is `fasterrcnn_resnet50_fpn_v2` (improved training recipe + heavier head).

## 7. References

- Ren, He, Girshick, Sun, *Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks*, NeurIPS 2015 — https://arxiv.org/abs/1506.01497
- Girshick, *Fast R-CNN*, ICCV 2015 — https://arxiv.org/abs/1504.08083
- Lin et al., *Feature Pyramid Networks for Object Detection*, CVPR 2017 — https://arxiv.org/abs/1612.03144
- torchvision detection models — https://pytorch.org/vision/stable/models.html#object-detection
- torchvision finetuning tutorial — https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html